<a href="https://colab.research.google.com/github/mdmostafizurrahman6351/Deep_Learning_Project/blob/main/project_03_electircal_betary_life_predction_using_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Electric vehical bettry life predction .
 using ANN

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

Data load


In [ ]:
url = 'https://raw.githubusercontent.com/mdmostafizurrahman6351/Maching_learning/refs/heads/main/Deep_Learning/Dataset/bettry_data.csv'
df = pd.read_csv(url)
print(df.shape)
df.head()

Data preprocessing

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df = df.drop(['battery_id',	'test_id',	'uid',	'filename', 'start_time'], axis=1)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df['Capacity'] = pd.to_numeric(df['Capacity'], errors='coerce')
df['Rct'] = pd.to_numeric(df['Rct'], errors='coerce')
df['Re'] = pd.to_numeric(df['Re'], errors='coerce')

df['Capacity'] = df.Capacity.fillna(df.Capacity.mean())
df['Rct'] = df.Rct.fillna(df.Rct.mean())
df['Re'] = df.Re.fillna(df.Re.mean())

df.isnull().sum()

In [ ]:
encoder = LabelEncoder()
df['type'] = encoder.fit_transform(df['type'])
df

train test split

In [ ]:
x = df.drop('Capacity', axis=1)
y = df['Capacity']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [ ]:
# scaling
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

model create

In [ ]:
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(x_train.shape[1],)))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='linear'))

model.summary()
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

model.fit(x_train, y_train, epochs=100, batch_size=32, validation_split=0.2)





In [ ]:
#evulate
loss, mae = model.evaluate(x_test, y_test)
print(f'Mean Absolute Error: {mae}')


In [ ]:
#save model
import joblib
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(encoder, 'encoder.pkl')
model.save('battery_life_prediction_model.h5')


create web application

In [ ]:
!pip install streamlit -q
!npm install -q -g localtunnel


In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import joblib
from tensorflow.keras.models import load_model

model = load_model('/content/battery_life_prediction_model.h5')
encoder = joblib.load('/content/encoder.pkl')
scaler = joblib.load('/content/scaler.pkl')

st.title("🔋 Battery Life Prediction")

type_input = st.selectbox("Select Type", ["charge", "discharge", "impedance"])
capacity = st.number_input("Enter Capacity", min_value=0.0, format="%.4f")
Rct = st.number_input("Enter Rct", min_value=0.0, format="%.4e")
Re = st.number_input("Enter Re", min_value=0.0, format="%.4e")

# predction function
def predict_life(type_val, cap, rct_val, re_val):
    # encoding
    type_encoded = encoder.transform([type_val])[0]

    x_input = np.array([[type_encoded, cap, re_val, rct_val]])

    # scaling
    x_input_scaled = scaler.transform(x_input)

    # predction
    prediction = model.predict(x_input_scaled)
    return prediction[0][0]

if st.button("Predict"):
    try:
        result = predict_life(type_input, capacity, Rct, Re)
        st.success(f'Predicted Battery Life: {result:.4f}')
    except Exception as e:
        st.error(f"Error: {e}")


In [ ]:
import urllib
print("Password/Enpoint IP for Tunnel: ", urllib.request.urlopen('https://ident.me').read().decode('utf8'))

!streamlit run app.py & npx localtunnel --port 8501
